Import modules

```markdown
### Instructions for Shared Users
To run this notebook successfully, please ensure you have a shortcut to the shared `world_cup_predictor` folder in your Google Drive:
1. Open the **'Shared with me'** tab in Google Drive.
2. Right-click the `world_cup_predictor` folder.
3. Select **'Organize'** > **'Add shortcut'**.
4. Choose **'My Drive'** and click **'Add'**.

This ensures the file path `/content/drive/MyDrive/world_cup_predictor/` is valid for your account.
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from pathlib import Path


Part 1: Reading the data (Alex)

In [ ]:
# Defines a base directory for the shared project folder
base_path = '/content/drive/MyDrive/world_cup_predictor/'

# Defines a directory and a list of filepaths consisting of relevant data
wc_data_dir = os.path.join(base_path, 'wc_data/')

filepaths = []

for f in os.listdir(wc_data_dir):
  filepaths.append(os.path.join(wc_data_dir, f))

In [ ]:
# The function convert_to_df reads each .csv file and returns a data frame
def convert_to_df(filepath):
  df = pd.read_csv(filepath)

  return df

In [ ]:
# Defines a dictionary that stores a data frame after the filepath is converted
dfs = {}

In [ ]:
# Loads all data frames from filepaths and stores them in dfs dictionary
for filepath in filepaths:
  filename = Path(filepath).stem

  df = convert_to_df(filepath)
  dfs[filename] = df

In [ ]:
# Displays first 10 rows of all data frames
for filename, df in dfs.items():
  print(f"Data Frame for {filename} file path (first 10 rows):")
  display(df.head(10))
  print(f"Number of rows in {filename}: {len(df)}\n\n")

Data Frame for fifa_ranking_2022-10-06 file path (first 10 rows):


,team,team_code,association,rank,previous_rank,points,previous_points
0,Brazil,BRA,CONMEBOL,1,1,1841.30,1837.56
1,Belgium,BEL,UEFA,2,2,1816.71,1821.92
2,Argentina,ARG,CONMEBOL,3,3,1773.88,1770.65
3,France,FRA,UEFA,4,4,1759.78,1764.85
4,England,ENG,UEFA,5,5,1728.47,1737.46
5,Italy,ITA,UEFA,6,7,1726.14,1713.86
6,Spain,ESP,UEFA,7,6,1715.22,1716.93
7,Netherlands,NED,UEFA,8,8,1694.51,1679.41
8,Portugal,POR,UEFA,9,9,1676.56,1678.65
9,Denmark,DEN,UEFA,10,10,1666.57,1665.47


Number of rows in fifa_ranking_2022-10-06: 211


Data Frame for matches_1930_2022 file path (first 10 rows):


,home_team,away_team,home_score,home_xg,home_penalty,away_score,away_xg,away_penalty,home_manager,home_captain,...,home_penalty_shootout_miss_long,away_penalty_shootout_miss_long,home_red_card,away_red_card,home_yellow_red_card,away_yellow_red_card,home_yellow_card_long,away_yellow_card_long,home_substitute_in_long,away_substitute_in_long
0,Argentina,France,3,3.3,4.0,3,2.2,2.0,Lionel Scaloni,Lionel Messi,...,NaN,"['3|1:1|Kingsley Coman', '5|2:1|Aurélien Tchou...",NaN,NaN,NaN,NaN,"['45+7&rsquor;|2:0|Enzo Fernández', '90+8&rsqu...","['55&rsquor;|2:0|Adrien Rabiot', '87&rsquor;|2...",['64&rsquor;|2:0|Marcos Acuña|for Ángel Di Mar...,['41&rsquor;|2:0|Randal Kolo Muani|for Ousmane...
1,Croatia,Morocco,2,0.7,NaN,1,1.2,NaN,Zlatko Dalić,Luka Modrić,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['69&rsquor;|2:1|Azzedine Ounahi', '84&rsquor;...",['61&rsquor;|2:1|Nikola Vlašić|for Andrej Kram...,['46&rsquor;|2:1|Ilias Chair|for Abdelhamid Sa...
2,France,Morocco,2,2.0,NaN,0,0.9,NaN,Didier Deschamps,Hugo Lloris,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,['27&rsquor;|1:0|Sofiane Boufal'],['65&rsquor;|1:0|Marcus Thuram|for Olivier Gir...,['21&rsquor;|1:0|Selim Amallah|for Romain Saïs...
3,Argentina,Croatia,3,2.3,NaN,0,0.5,NaN,Lionel Scaloni,Lionel Messi,...,NaN,NaN,NaN,NaN,NaN,NaN,"['68&rsquor;|2:0|Cristian Romero', '71&rsquor;...","['32&rsquor;|0:0|Mateo Kovačić', '32&rsquor;|0...",['62&rsquor;|2:0|Lisandro Martínez|for Leandro...,"['46&rsquor;|2:0|Mislav Oršić|for Borna Sosa',..."
4,Morocco,Portugal,1,1.4,NaN,0,0.9,NaN,Hoalid Regragui,Romain Saïss,...,NaN,NaN,NaN,NaN,Walid Cheddira · 90+3,NaN,"['70&rsquor;|1:0|Achraf Dari', '90+1&rsquor;|1...",['87&rsquor;|1:0|Vitinha'],['57&rsquor;|1:0|Achraf Dari|for Romain Saïss'...,['51&rsquor;|1:0|João Cancelo|for Raphaël Guer...
5,England,France,1,2.4,NaN,2,0.9,NaN,Gareth Southgate,Harry Kane,...,NaN,NaN,NaN,NaN,NaN,NaN,['90&rsquor;|1:2|Harry Maguire'],"['43&rsquor;|0:1|Antoine Griezmann', '47&rsquo...",['79&rsquor;|1:2|Raheem Sterling|for Bukayo Sa...,['79&rsquor;|1:2|Kingsley Coman|for Ousmane De...
6,Croatia,Brazil,1,0.6,4.0,1,2.5,2.0,Zlatko Dalić,Luka Modrić,...,NaN,"['2|1:0|Rodrygo', '8|4:2|Marquinhos']",NaN,NaN,NaN,NaN,"['31&rsquor;|0:0|Marcelo Brozović', '117&rsquo...","['25&rsquor;|0:0|Danilo', '68&rsquor;|0:0|Case...",['72&rsquor;|0:0|Nikola Vlašić|for Mario Pašal...,"['56&rsquor;|0:0|Antony|for Raphinha', '64&rsq..."
7,Netherlands,Argentina,2,0.6,3.0,2,1.9,4.0,Louis van Gaal,Virgil van Dijk,...,"['1|0:0|Virgil van Dijk', '3|0:1|Steven Berghu...",['8|2:3|Enzo Fernández'],NaN,NaN,Denzel Dumfries · 120+3,NaN,"['43&rsquor;|0:1|Jurriën Timber', '45+2&rsquor...","['31&rsquor;|0:0|Walter Samuel', '43&rsquor;|0...",['46&rsquor;|0:1|Steven Berghuis|for Steven Be...,['66&rsquor;|0:1|Leandro Paredes|for Rodrigo D...
8,Morocco,Spain,0,0.7,3.0,0,1.0,0.0,Hoalid Regragui,Romain Saïss,...,['5|2:0|Badr Banoun'],"['2|1:0|Pablo Sarabia', '4|2:0|Carlos Soler', ...",NaN,NaN,NaN,NaN,['90&rsquor;|0:0|Romain Saïss'],['77&rsquor;|0:0|Aymeric Laporte'],['66&rsquor;|0:0|Abdessamad Ezzalzouli|for Sof...,['63&rsquor;|0:0|Álvaro Morata|for Marco Asens...
9,Portugal,Switzerland,6,2.3,NaN,1,1.2,NaN,Fernando Santos,Pepe,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['43&rsquor;|2:0|Fabian Schär', '59&rsquor;|4:...",['73&rsquor;|5:1|Cristiano Ronaldo|for Gonçalo...,['46&rsquor;|2:0|Eray Cömert|for Fabian Schär'...


Number of rows in matches_1930_2022: 964


Data Frame for schedule_2026 file path (first 10 rows):


...

Number of rows in schedule_2026: 72


Data Frame for fifa_ranking_2026-06-08 file path (first 10 rows):


,team,team_code,association,rank,previous_rank,points,previous_points,rated_matches
0,Argentina,ARG,CONMEBOL,1,3,1876.118331,1874.814835,59
1,Spain,ESP,UEFA,2,2,1873.013187,1876.395199,56
2,France,FRA,UEFA,3,1,1869.428449,1877.322731,57
3,England,ENG,UEFA,4,4,1827.048678,1825.965482,57
4,Portugal,POR,UEFA,5,5,1766.177547,1763.834406,56
5,Brazil,BRA,CONMEBOL,6,6,1765.856297,1761.160930,55
6,Morocco,MAR,CAF,7,8,1755.100232,1755.868410,87
7,Netherlands,NED,UEFA,8,7,1751.097835,1757.874428,56
8,Belgium,BEL,UEFA,9,9,1742.235945,1734.714832,54
9,Germany,GER,UEFA,10,10,1735.771984,1730.371360,55


Number of rows in fifa_ranking_2026-06-08: 211


Data Frame for world_cup file path (first 10 rows):


,Year,Host,Teams,Champion,Runner-Up,TopScorrer,Attendance,AttendanceAvg,Matches
0,2022,Qatar,32,Argentina,France,Kylian Mbappé - 8,3404252,53191,64
1,2018,Russia,32,France,Croatia,Harry Kane - 6,3031768,47371,64
2,2014,Brazil,32,Germany,Argentina,James Rodríguez - 6,3429873,53592,64
3,2010,South Africa,32,Spain,Netherlands,"Wesley Sneijder, Thomas Müller... - 5",3178856,49670,64
4,2006,Germany,32,Italy,France,Miroslav Klose - 5,3352605,52384,64
5,2002,"Korea Republic, Japan",32,Brazil,Germany,Ronaldo - 8,2705337,42271,64
6,1998,France,32,France,Brazil,Davor Šuker - 6,2903477,45367,64
7,1994,United States,24,Brazil,Italy,"Hristo Stoichkov, Oleg Salenko - 6",3587538,68991,52
8,1990,Italy,24,West Germany,Argentina,Salvatore Schillaci - 6,2516215,48389,52
9,1986,Mexico,24,Argentina,West Germany,Gary Lineker - 6,2394031,46039,52


Number of rows in world_cup: 22




Part 2: Train-test split

**Sources:**

*Part 1:*

https://www.geeksforgeeks.org/python/os-module-python-examples/

https://www.geeksforgeeks.org/python/python-program-to-get-the-file-name-from-the-file-path/



*Part 2:*

**Insert sources here**

*Overall:*

Code Reference: https://www.kaggle.com/code/ahmedwaleed1903/world-cup

Data: https://www.kaggle.com/datasets/piterfm/fifa-football-world-cup?resource=download

**AI Usage**

Google Gemini (Gemini 3 Flash) was used to write and debug some specific lines/blocks of code in the notebook.